# Optuna + observabilidad: `resized_672_swint`

Lleva al límite una sola celda de la matriz de `multirun.ipynb` (`resized_672_swint_seed42`, etiqueta A) sin tocar `src/`:

1. **Búsqueda con Optuna** (~12 h de A100, ~20 trials con poda) sobre LR por etapa, cabeza, regularización, pérdida, aumento, CLAHE y MixUp de positivos. El objetivo es la PR-AUC por imagen en el val canónico (tarea A); test no se mira durante la búsqueda.
2. **Confirmación**: el mejor config con las etiquetas A, B y C × semillas 42/43/44 (8 corridas nuevas, ~8 h), siempre evaluado sobre la tarea A y las mismas imágenes, y comparado contra el baseline `resized_672_swint_seed{42,43,44}` de `tesis_v4`.
3. **Exprimir sin reentrenar** (decidido en val): TTA, ensembles, agregación por mama máximo vs media y temperature scaling.
4. **Observabilidad**: Grad-CAM / Grad-CAM++ / HiResCAM, Integrated Gradients, SmoothGrad y oclusión; métricas de localización contra la ROI anotada, chequeo de atajos, sanity check por aleatorización, t-SNE de embeddings, galería de errores y análisis por subgrupos.
5. **Reporte final en test**, una sola vez.

Los hiperparámetros que `src` tiene fijos en el código (LR por etapa, `top_dense`, `dropout`, optimizador, label smoothing, magnitudes de aumento, CLAHE, MixUp) se inyectan con **hooks en runtime** (monkeypatch desde este notebook). Viajan como claves extra de `CONFIG["TRAINING"]`, que `src` ignora y loguea en Comet como parámetros. Con los valores por defecto, los hooks reproducen exactamente el comportamiento de `src`.

Todo va al proyecto de Comet `tesis_v4_hpo` (la matriz de `tesis_v4` no se toca, solo se lee el baseline). Nada se persiste en el disco de Colab: el estudio de Optuna es un SQLite temporal que se sincroniza como asset `optuna.db` del experimento `optuna_estudio`. Si se cae la VM, volver a ejecutar el notebook entero: el estudio sigue, el trial cortado se reintenta con los mismos parámetros y retoma desde su última época (mismo nombre, mismo experimento de Comet).

In [ ]:
!rm -rf ./src && git clone https://github.com/emiliodelgadouy/TesisV4_src src
!pip install -q optuna
from src.notebook import configure_notebook

configure_notebook(autoreload=False)  # sin autoreload: recargar src borraría los hooks de HPO
from src.notebook_api import *

In [ ]:
import copy
import hashlib
import json
import math
import tempfile
import time
from dataclasses import dataclass
from pathlib import Path

import comet_ml
import matplotlib.pyplot as plt
import optuna
from IPython.display import display
from tensorflow import keras

from src.dataset.provider import as_tf_dataset
from src.training.evaluator import BreastAggregation, ClinicalMetrics, PairedBootstrap, ThresholdSelector
from src.training.plan import log_table_asset
from src.training.resources import GpuResources

# Valores que src tiene fijos en el código: con estos defaults los hooks no cambian nada.
HPO_DEFAULTS = {
    "LR_HEAD": 1e-3,  # etapa 1 (factory.py)
    "LR_PARTIAL": 1e-4,  # etapa 2 (experiment.py)
    "LR_FULL": 1e-5,  # etapa 3 (experiment.py)
    "TOP_DENSE": 256,
    "DROPOUT": 0.4,
    "REDUCE_LR_FACTOR": 0.5,
    "MIN_LR": 1e-7,
    "WEIGHT_DECAY": 0.0,  # > 0 cambia Adam por AdamW (con clipnorm=1)
    "LABEL_SMOOTHING": 0.0,
    "AUG_STRENGTH": 1.0,  # multiplica las magnitudes del aumento agresivo
    "CLAHE_CLIP": 2.0,
    "CLAHE_TILE": 8,
    "POSITIVE_MIXUP": False,
    "MIXUP_ALPHA": 0.2,
    "MIXUP_PROBABILITY": 0.5,
}

GENERAL = {
    "RANDOM_SEED": 42,
    "REDUCED_DATASET": False,
    "USE_CLAHE": True,
    "TARGET_MODE": "birads",
    "VALIDATION_SPLIT_RATIO": 0.20,
    "PROBABILITY_THRESHOLD": 0.5,
    "NO_FINDING_TO_FINDING_RATIO": 3,
    "METRIC_TO_MAXIMIZE": "pr_auc",
    "BATCH_SIZE": 32,
    "USE_CUSTOM_BATCH_SIZE": True,
    "CACHE_DATASET": True,
    "REGENERATE_SPLITS": False,
}
TRAINING = {
    "EPOCHS_FROZEN_BACKBONE": 15,
    "EPOCHS_PARTIAL_BACKBONE": 25,
    "EPOCHS_FULL_FINETUNE": 15,
    "FOCAL_GAMMA": 2.0,
    "FOCAL_ALPHA": None,  # None: n_neg / n del train (lo calcula src)
    "AGGRESSIVE_AUGMENTATION": True,
    "BACKBONE_TRAINABLE_FRACTION": 0.30,
    "EARLY_STOPPING_PATIENCE": 4,
    "REDUCE_LR_PATIENCE": 2,
    **HPO_DEFAULTS,
}
ABMIL = {
    "BAG_GRID": (3, 3),
    "BAG_CANVAS_MODE": "resize",
    "BAG_KERAS_TILING": True,
    "BATCH_SIZE": 32,
    "ATTENTION_DIM": 128,
    "ATTENTION_GATED": True,
}
PATCH = {
    "POSITIVE": 1,
    "HARD_NEGATIVE": 0,
    "RANDOM_NEGATIVE": GENERAL["NO_FINDING_TO_FINDING_RATIO"],
    "RESIZE_TO_BAG_CANVAS": True,
}
PATCH_HARDNEG = {
    "POSITIVE": 1,
    "HARD_NEGATIVE": GENERAL["NO_FINDING_TO_FINDING_RATIO"] / 2,
    "RANDOM_NEGATIVE": GENERAL["NO_FINDING_TO_FINDING_RATIO"] / 2,
    "ALIGN_TO_BAG_GRID": True,
}
COMET = {
    "API_KEY": "SzBk2RvzTzZCQMpLNBh35mg1q",
    "PROJECT_NAME": "tesis-v4-optuna",  # separado de la matriz (tesis_v4)
    "N_SAMPLE_IMAGES": 8,
}
MODELS = ["swint"]
RESIZED = {
    "INPUT_SIZES": [(672, 672)],
    "BATCH_SIZES": {672: 64},
    "CACHE": {672: True},
}
RESUME = {
    "ENABLED": True,
    "BACKEND": "comet",
    "DIR": "runs",
    "SKIP_IF_IN_COMET": True,
}
PLAN = {
    "TASK": "A",  # tarea de evaluación fija: 1-2 vs 3-5
    "BOOTSTRAP_SAMPLES": 1000,
    "STATUS_EXPERIMENT": "estado_hpo",
}
HPO = {
    "STUDY_NAME": "resized_672_swint_hpo",
    "STUDY_EXPERIMENT": "optuna_estudio",  # experimento de Comet con el asset optuna.db
    "N_TRIALS": 20,  # trials terminados (completos + podados)
    "HOURS": 12.0,  # presupuesto de la búsqueda (suma de horas de los trials)
    "SEED": 42,
}
CONFIG = {
    "GENERAL": GENERAL,
    "TRAINING": TRAINING,
    "ABMIL": ABMIL,
    "COMET": COMET,
    "MODELS": MODELS,
    "PATCH": PATCH,
    "PATCH_HARDNEG": PATCH_HARDNEG,
    "RESIZED": RESIZED,
    "RESUME": RESUME,
    "PLAN": PLAN,
}

# Baseline de la matriz: misma config que multirun, proyecto tesis_v4, sin claves HPO.
BASE_CONFIG = copy.deepcopy(CONFIG)
BASE_CONFIG["COMET"]["PROJECT_NAME"] = "tesis_v4"
BASE_CONFIG["PLAN"]["STATUS_EXPERIMENT"] = "estado_matriz"
for key in [*HPO_DEFAULTS, "FOCAL_ALPHA"]:
    BASE_CONFIG["TRAINING"].pop(key, None)

set_random_seeds(CONFIG)
login_comet(CONFIG)
prepare_resume(CONFIG)

## Dataset

Los mismos splits canónicos que `multirun.ipynb` (`splits/dataset_splits.json`, sin regenerar).

In [ ]:
ds = prepare_dataset(CONFIG)
train, val, test = get_dataset_splits(CONFIG, ds)

## Hooks de HPO (sin tocar `src`)

Reemplazan en runtime los nombres que usa el pipeline. Leen los valores de `config["TRAINING"]` y, si falta una clave, usan `HPO_DEFAULTS` (= lo que hace `src`):

| Hook | Qué inyecta |
| --- | --- |
| `experiment.ModelBuilder`, `inference.ModelBuilder` | `top_dense`, `dropout`, LR de la etapa 1, `reduce_lr_factor`, `min_lr`; deja `builder._hpo` |
| `BaseModelBuilder.make_backbone_partially_trainable` / `make_backbone_trainable` | LR de las etapas 2 y 3 en lugar de 1e-4 / 1e-5 |
| `BaseModelBuilder.optimizer` | `AdamW` (sin decay en normas, bias ni tablas de posición) + `clipnorm=1` si `WEIGHT_DECAY > 0` |
| `BaseModelBuilder.focal_loss` | `label_smoothing` |
| `BaseModelBuilder.augmentation_seq` | mismas capas y nombres, magnitudes × `AUG_STRENGTH` |
| `experiment.build_dataset_provider`, `inference.build_dataset_provider` | CLAHE (`clip`, `tile`) y MixUp de positivos; train y re-predicción usan el mismo CLAHE |
| `ModelTrainer.callbacks` | poda de Optuna cuando hay un trial activo (antes del `ResumeCallback`, que sigue último) |

La celda es idempotente: guarda los originales la primera vez y siempre parchea a partir de ellos.

In [ ]:
import src.dataset.provider as _provider_mod
import src.training.experiment as _experiment_mod
import src.training.inference as _inference_mod
from src.model_builder.base import BaseModelBuilder
from src.model_builder.factory import ModelBuilderFactory
from src.training.model_trainer import ModelTrainer

_HPO_ORIGINALS = globals().get("_HPO_ORIGINALS") or {
    "create": ModelBuilderFactory.create,
    "partial": BaseModelBuilder.make_backbone_partially_trainable,
    "full": BaseModelBuilder.make_backbone_trainable,
    "optimizer": BaseModelBuilder.optimizer,
    "focal_loss": BaseModelBuilder.focal_loss,
    "augmentation_seq": BaseModelBuilder.augmentation_seq,
    "callbacks": ModelTrainer.callbacks,
    "provider": _provider_mod.build_dataset_provider,
}
PODA = {"trial": None, "best": float("-inf")}


def hpo_values(config) -> dict:
    training = (config or {}).get("TRAINING") or {}
    return {key: training.get(key, default) for key, default in HPO_DEFAULTS.items()}


def create_with_hpo(config, IMG_SIZE, backbone, preprocess_input, **kwargs):
    hpo = hpo_values(config)
    kwargs.setdefault("top_dense", int(hpo["TOP_DENSE"]))
    kwargs.setdefault("dropout", float(hpo["DROPOUT"]))
    kwargs.setdefault("learning_rate", float(hpo["LR_HEAD"]))
    kwargs.setdefault("reduce_lr_factor", float(hpo["REDUCE_LR_FACTOR"]))
    kwargs.setdefault("min_lr", float(hpo["MIN_LR"]))
    builder = _HPO_ORIGINALS["create"](config, IMG_SIZE, backbone, preprocess_input, **kwargs)
    builder._hpo = hpo
    return builder


def partial_with_hpo(self, trainable_fraction=0.30, learning_rate=None, train_batch_norm=False):
    hpo = getattr(self, "_hpo", None)
    if hpo is not None:
        learning_rate = float(hpo["LR_PARTIAL"])
    return _HPO_ORIGINALS["partial"](self, trainable_fraction=trainable_fraction, learning_rate=learning_rate, train_batch_norm=train_batch_norm)


def full_with_hpo(self, trainable=True, learning_rate=None, train_batch_norm=False):
    hpo = getattr(self, "_hpo", None)
    if hpo is not None and trainable:
        learning_rate = float(hpo["LR_FULL"])
    return _HPO_ORIGINALS["full"](self, trainable=trainable, learning_rate=learning_rate, train_batch_norm=train_batch_norm)


def optimizer_with_hpo(self):
    hpo = getattr(self, "_hpo", None)
    weight_decay = float(hpo["WEIGHT_DECAY"]) if hpo is not None else 0.0
    if weight_decay <= 0:
        return _HPO_ORIGINALS["optimizer"](self)
    optimizer = keras.optimizers.AdamW(learning_rate=self.learning_rate, weight_decay=weight_decay, clipnorm=1.0)
    try:
        optimizer.exclude_from_weight_decay(var_names=["bias", "norm", "relative_position_bias_table"])
    except Exception as exc:
        print(f"[hpo] AdamW sin exclusiones de weight decay ({exc})")
    return optimizer


def focal_loss_with_hpo(self):
    hpo = getattr(self, "_hpo", None)
    smoothing = float(hpo["LABEL_SMOOTHING"]) if hpo is not None else 0.0
    if smoothing <= 0:
        return _HPO_ORIGINALS["focal_loss"](self)
    return keras.losses.BinaryFocalCrossentropy(
        apply_class_balancing=True,
        alpha=self.focal_alpha,
        gamma=self.focal_gamma,
        from_logits=self.loss_from_logits,
        label_smoothing=smoothing,
    )


def augmentation_seq_with_hpo(self):
    hpo = getattr(self, "_hpo", None)
    strength = float(hpo["AUG_STRENGTH"]) if hpo is not None else 1.0
    if not self.aggressive_augmentation or strength == 1.0:
        return _HPO_ORIGINALS["augmentation_seq"](self)
    layers = keras.layers
    sequence = []
    if not self.lateralized_inputs:
        sequence.append(layers.RandomFlip("horizontal", name="aug_flip_h"))
    zoom = min(0.22 * strength, 0.5)
    sequence.extend(
        [
            layers.RandomRotation(min(0.14 * strength, 0.5), fill_mode="reflect", name="aug_rot"),
            layers.RandomZoom(height_factor=(0.0, zoom), width_factor=(0.0, zoom), fill_mode="reflect", name="aug_zoom"),
            layers.RandomTranslation(height_factor=min(0.14 * strength, 0.5), width_factor=min(0.14 * strength, 0.5), fill_mode="reflect", name="aug_translate"),
            layers.RandomContrast(min(0.25 * strength, 0.9), name="aug_contrast"),
            layers.RandomBrightness(min(0.25 * strength, 0.9), value_range=(0.0, 255.0), name="aug_brightness"),
        ]
    )
    return keras.Sequential(sequence, name="augmentation_aggressive")


def provider_with_hpo(config=None, image_size=None, batch_size=None, *, provider_config=None, **kwargs):
    if provider_config is None and config is not None:
        hpo = hpo_values(config)
        kwargs.setdefault("clahe_clip_limit", float(hpo["CLAHE_CLIP"]))
        kwargs.setdefault("clahe_tile_grid", int(hpo["CLAHE_TILE"]))
        if hpo["POSITIVE_MIXUP"]:
            kwargs.setdefault("positive_mixup", True)
            kwargs.setdefault("positive_mixup_alpha", float(hpo["MIXUP_ALPHA"]))
            kwargs.setdefault("positive_mixup_probability", float(hpo["MIXUP_PROBABILITY"]))
    return _HPO_ORIGINALS["provider"](config, image_size, batch_size, provider_config=provider_config, **kwargs)


def _running_comet_experiment():
    for getter in ("get_running_experiment", "get_global_experiment"):
        fn = getattr(comet_ml, getter, None)
        if fn is not None:
            try:
                experiment = fn()
            except Exception:
                experiment = None
            if experiment is not None:
                return experiment
    return None


class OptunaPruning(keras.callbacks.Callback):
    """Reporta el máximo acumulado de la métrica monitoreada por época global y poda si Optuna lo pide."""

    def __init__(self, trainer, trial, monitor):
        super().__init__()
        self.trainer = trainer
        self.trial = trial
        self.monitor = monitor

    def on_epoch_end(self, epoch, logs=None):
        value = (logs or {}).get(self.monitor)
        if value is None or not math.isfinite(float(value)):
            return
        PODA["best"] = max(PODA["best"], float(value))
        step = int(self.trainer.epoch_offset) + int(epoch) + 1
        self.trial.report(PODA["best"], step)
        if self.trial.should_prune():
            experiment = _running_comet_experiment()
            if experiment is not None:
                try:
                    experiment.log_other("hpo_status", "pruned")
                    experiment.log_other("hpo_pruned_at_epoch", step)
                except Exception:
                    pass
            raise optuna.TrialPruned(f"podado en la época global {step} ({self.monitor}={PODA['best']:.4f})")


def callbacks_with_hpo(self, training_timer=None):
    callbacks = _HPO_ORIGINALS["callbacks"](self, training_timer=training_timer)
    if PODA["trial"] is not None:
        callbacks.append(OptunaPruning(self, PODA["trial"], self.builder.checkpoint_monitor))
    return callbacks


_experiment_mod.ModelBuilder = create_with_hpo
_inference_mod.ModelBuilder = create_with_hpo
_experiment_mod.build_dataset_provider = provider_with_hpo
_inference_mod.build_dataset_provider = provider_with_hpo
BaseModelBuilder.make_backbone_partially_trainable = partial_with_hpo
BaseModelBuilder.make_backbone_trainable = full_with_hpo
BaseModelBuilder.optimizer = optimizer_with_hpo
BaseModelBuilder.focal_loss = focal_loss_with_hpo
BaseModelBuilder.augmentation_seq = augmentation_seq_with_hpo
ModelTrainer.callbacks = callbacks_with_hpo
print("[hpo] hooks activos")

### Prueba rápida de los hooks

Arma el grafo de `resized_672_swint` con valores no default (sin entrenar ni leer imágenes) y verifica que se aplicaron. Tarda lo que tarda construir Swin-T; `PROBAR_HOOKS=False` la saltea.

In [ ]:
PROBAR_HOOKS = True

if PROBAR_HOOKS:
    from src.backbones import resolve_backbone

    _cfg = copy.deepcopy(CONFIG)
    _cfg["TRAINING"].update({"TOP_DENSE": 128, "DROPOUT": 0.25, "LR_HEAD": 5e-4, "LR_PARTIAL": 7e-5, "LR_FULL": 3e-6, "WEIGHT_DECAY": 1e-4, "LABEL_SMOOTHING": 0.05, "AUG_STRENGTH": 1.3, "CLAHE_CLIP": 3.0})
    _backbone, _preprocess, _size = resolve_backbone("swint", input_size=(672, 672))
    _builder = _experiment_mod.ModelBuilder(_cfg, _size, _backbone, _preprocess, mode="resized", initial_bias=-1.0, focal_alpha=0.75, lateralized_inputs=True, steps_per_execution=1)
    _builder.build()

    def _lr(builder):
        value = builder.model.optimizer.learning_rate
        try:
            return float(np.asarray(value))
        except Exception:
            return float(value.numpy())

    def _factor(layer):
        factor = layer.get_config()["factor"]
        return float(max(abs(v) for v in factor)) if isinstance(factor, (list, tuple)) else float(factor)

    checks = {
        "top_dense": _builder.model.get_layer("dense").units == 128,
        "dropout": abs(_builder.model.get_layer("dropout").rate - 0.25) < 1e-9,
        "adamw": isinstance(getattr(_builder.model.optimizer, "inner_optimizer", _builder.model.optimizer), keras.optimizers.AdamW),
        "lr_head": abs(_lr(_builder) - 5e-4) < 1e-9,
        "label_smoothing": abs(float(_builder.focal_loss().get_config().get("label_smoothing", 0.0)) - 0.05) < 1e-9,
        "aug_rot": abs(_factor(_builder.augmentation_seq().get_layer("aug_rot")) - 0.14 * 1.3) < 1e-6,
    }
    _builder.make_backbone_partially_trainable(trainable_fraction=0.3, learning_rate=1e-4)
    checks["lr_partial"] = abs(_lr(_builder) - 7e-5) < 1e-9
    _builder.make_backbone_trainable(trainable=True, learning_rate=1e-5)
    checks["lr_full"] = abs(_lr(_builder) - 3e-6) < 1e-9
    _provider = _experiment_mod.build_dataset_provider(_cfg, _size, 4, lateralize=True, mode="resized", cache_dataset=False)
    checks["clahe_clip"] = abs(float(_provider.config.clahe_clip_limit) - 3.0) < 1e-9
    display(pd.Series(checks, name="ok"))
    GpuResources.dispose_model_builder(_builder)
    del _builder, _backbone, _provider
    GpuResources.release(clear_keras_session=True)
    assert all(checks.values()), f"hooks que no se aplicaron: {[k for k, ok in checks.items() if not ok]}"

## `HpoSpec` y `HpoPlan`

`HpoSpec` es un `RunSpec` con los hiperparámetros del trial. El nombre del experimento lleva un hash de esos parámetros (`resized_672_swint_hpo1a2b3c4d_seed42`, más `_lblb` / `_lblc` para otras etiquetas): la misma combinación siempre cae en el mismo experimento de Comet, así que un trial reintentado retoma desde su última época. `HpoPlan` mezcla esos parámetros en `CONFIG["TRAINING"]` de cada corrida; todo lo demás (resume, salteo de lo terminado, `predictions_{split}.csv` canónicos, `evaluate`, `seed_summary`, tabla de estado) es el `ExperimentPlan` de `src`.

In [ ]:
def normalizar(params) -> tuple:
    """Parámetros como tupla ordenada y estable (floats a 4 cifras significativas) para nombrar y hashear."""
    out = []
    for key in sorted(params):
        value = params[key]
        if isinstance(value, (bool, np.bool_)):
            value = bool(value)
        elif isinstance(value, (int, np.integer)):
            value = int(value)
        elif isinstance(value, (float, np.floating)):
            value = float(f"{float(value):.4g}")
        out.append((key, value))
    return tuple(out)


@dataclass(frozen=True)
class HpoSpec(RunSpec):
    params: tuple = ()

    @property
    def tag(self) -> str | None:
        if not self.params:
            return None
        return "hpo" + hashlib.sha1(json.dumps(self.params, default=str).encode("utf-8")).hexdigest()[:8]

    @property
    def suffix(self) -> str | None:
        parts = [part for part in (self.tag, RunSpec.suffix.fget(self)) if part]
        return "_".join(parts) or None


class HpoPlan(ExperimentPlan):
    def _config_for(self, spec):
        config = super()._config_for(spec)
        config["TRAINING"].update(dict(getattr(spec, "params", ()) or ()))
        return config


def spec_de(params, *, seed=None, label=None) -> HpoSpec:
    return HpoSpec("resized", "swint", input_size=672, seed=seed, label=label, params=normalizar(params))


def metricas(plan_, spec, split="val", level="image") -> dict:
    """Métricas clínicas de la tarea A sobre el split canónico; umbral de Youden elegido en val al mismo nivel."""
    y, prob, _ = plan_._level_arrays(plan_.predictions(spec, split), plan_.task, level)
    y_val, prob_val, _ = plan_._level_arrays(plan_.predictions(spec, "val"), plan_.task, level)
    return ClinicalMetrics.compute(y, prob, threshold=ThresholdSelector.youden_j(y_val, prob_val))


plan = HpoPlan(CONFIG, train, val, test)
BASE_PLAN = ExperimentPlan(BASE_CONFIG, train, val, test)
SEMILLAS = [42, 43, 44]
ETIQUETAS = ["A", "B", "C"]
BASE_SPECS = [RunSpec("resized", "swint", input_size=672, seed=seed) for seed in SEMILLAS]
display(plan.check_splits().pivot(index="label", columns="split", values=["n", "n_pos", "prevalence"]).round(4))
display(BASE_PLAN.status_table(BASE_SPECS))

## Estudio de Optuna

- **Storage**: SQLite en un directorio temporal, bajado del asset `optuna.db` del experimento `optuna_estudio` (tag `hpo`) al arrancar y subido al empezar y al terminar cada trial. Con heartbeat: si la VM muere, el trial que quedó `RUNNING` pasa a `FAIL` al retomar y `RetryFailedTrialCallback` lo reencola con los mismos parámetros (mismo experimento de Comet, retoma desde la última época).
- **Sampler**: TPE multivariado con grupos (maneja los parámetros condicionales), semilla 42, 6 trials aleatorios iniciales. El trial 0 es el baseline (los defaults) bajo el mismo pipeline.
- **Poda**: `MedianPruner` sobre el máximo acumulado de `val_pr_auc` por época global; no poda durante la etapa congelada ni antes de 5 trials.
- **Objetivo**: PR-AUC por imagen del val canónico, tarea A (`plan.predictions`). Se guardan como atributos ROC-AUC, PR-AUC por mama, sensibilidad a especificidad 90 %, horas y la URL de Comet.

In [ ]:
import sqlite3

from optuna.storages import RDBStorage, RetryFailedTrialCallback
from optuna.study import MaxTrialsCallback
from optuna.trial import TrialState


class CometStudyStore:
    """SQLite temporal de Optuna sincronizado como asset de un experimento de Comet."""

    def __init__(self, plan_, experiment_name: str, asset_name: str = "optuna.db") -> None:
        self.index = plan_.comet
        self.experiment_name = experiment_name
        self.asset_name = asset_name
        self.dir = Path(tempfile.mkdtemp(prefix="optuna_"))
        self.path = self.dir / asset_name
        self._exp = None

    @property
    def url(self) -> str:
        return f"sqlite:///{self.path}"

    def experiment(self):
        if self._exp is None:
            found = self.index.experiments_named(self.experiment_name, fresh=True)
            if found:
                self._exp = found[0]
            else:
                workspace, project = self.index._resolve_project()
                self._exp = comet_ml.APIExperiment(api_key=self.index.api_key, workspace=workspace, project_name=project, experiment_name=self.experiment_name)
                self._exp.add_tag("hpo")
                self.index.forget(self.experiment_name)
        return self._exp

    def download(self) -> bool:
        assets = [asset for asset in self.experiment().get_asset_list() or [] if asset.get("fileName") == self.asset_name]
        if not assets:
            print(f"[hpo] {self.experiment_name} no tiene {self.asset_name}: estudio nuevo")
            return False
        newest = max(assets, key=lambda asset: int(asset.get("createdAt") or 0))
        self.path.write_bytes(self.experiment().get_asset(newest["assetId"], return_type="binary"))
        print(f"[hpo] estudio bajado de Comet ({self.experiment_name}/{self.asset_name})")
        return True

    def upload(self, *_args) -> None:
        """Copia consistente (API de backup de SQLite) y la sube pisando el asset; un error de red no corta el estudio."""
        snapshot = self.dir / f"snapshot_{self.asset_name}"
        try:
            source = sqlite3.connect(self.path)
            target = sqlite3.connect(snapshot)
            try:
                source.backup(target)
            finally:
                target.close()
                source.close()
            self.experiment().log_asset(str(snapshot), name=self.asset_name, overwrite=True)
        except Exception as exc:
            print(f"[hpo] no pude subir {self.asset_name} a Comet ({type(exc).__name__}: {exc})")
        finally:
            snapshot.unlink(missing_ok=True)


STORE = CometStudyStore(plan, HPO["STUDY_EXPERIMENT"])
STORE.download()
storage = RDBStorage(
    STORE.url,
    heartbeat_interval=60,
    grace_period=180,
    failed_trial_callback=RetryFailedTrialCallback(max_retry=1),
)
study = optuna.create_study(
    study_name=HPO["STUDY_NAME"],
    storage=storage,
    direction="maximize",
    load_if_exists=True,
    sampler=optuna.samplers.TPESampler(multivariate=True, group=True, seed=HPO["SEED"], n_startup_trials=6),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=TRAINING["EPOCHS_FROZEN_BACKBONE"]),
)

# Trial 0: el baseline (defaults de src) bajo el mismo pipeline.
TRIAL_BASELINE = {
    "LR_HEAD": 1e-3,
    "LR_PARTIAL": 1e-4,
    "LR_FULL": 1e-5,
    "BACKBONE_TRAINABLE_FRACTION": 0.30,
    "TOP_DENSE": 256,
    "DROPOUT": 0.4,
    "LABEL_SMOOTHING": 0.0,
    "FOCAL_GAMMA": 2.0,
    "AUG_STRENGTH": 1.0,
    "CLAHE_CLIP": 2.0,
    "EARLY_STOPPING_PATIENCE": 4,
    "REDUCE_LR_PATIENCE": 2,
    "USE_WEIGHT_DECAY": False,
    "FOCAL_ALPHA_AUTO": True,
    "POSITIVE_MIXUP": False,
}
if not study.trials:
    study.enqueue_trial(TRIAL_BASELINE)


def sugerir(trial) -> dict:
    """Espacio de búsqueda (etiqueta A, semilla 42, canvas 672, batch 64 fijo)."""
    params = {
        "LR_HEAD": trial.suggest_float("LR_HEAD", 3e-4, 3e-3, log=True),
        "LR_PARTIAL": trial.suggest_float("LR_PARTIAL", 2e-5, 3e-4, log=True),
        "LR_FULL": trial.suggest_float("LR_FULL", 2e-6, 3e-5, log=True),
        "BACKBONE_TRAINABLE_FRACTION": trial.suggest_float("BACKBONE_TRAINABLE_FRACTION", 0.15, 0.6),
        "TOP_DENSE": trial.suggest_categorical("TOP_DENSE", [128, 256, 512]),
        "DROPOUT": trial.suggest_float("DROPOUT", 0.1, 0.6),
        "LABEL_SMOOTHING": trial.suggest_float("LABEL_SMOOTHING", 0.0, 0.1),
        "FOCAL_GAMMA": trial.suggest_float("FOCAL_GAMMA", 0.0, 3.0),
        "AUG_STRENGTH": trial.suggest_float("AUG_STRENGTH", 0.5, 1.5),
        "CLAHE_CLIP": trial.suggest_float("CLAHE_CLIP", 1.0, 4.0),
        "EARLY_STOPPING_PATIENCE": trial.suggest_categorical("EARLY_STOPPING_PATIENCE", [4, 6]),
        "REDUCE_LR_PATIENCE": trial.suggest_categorical("REDUCE_LR_PATIENCE", [2, 3]),
    }
    use_wd = trial.suggest_categorical("USE_WEIGHT_DECAY", [False, True])
    params["WEIGHT_DECAY"] = trial.suggest_float("WEIGHT_DECAY", 1e-5, 5e-2, log=True) if use_wd else 0.0
    alpha_auto = trial.suggest_categorical("FOCAL_ALPHA_AUTO", [True, False])
    params["FOCAL_ALPHA"] = None if alpha_auto else trial.suggest_float("FOCAL_ALPHA", 0.5, 0.9)
    params["POSITIVE_MIXUP"] = trial.suggest_categorical("POSITIVE_MIXUP", [False, True])
    params["MIXUP_ALPHA"] = trial.suggest_float("MIXUP_ALPHA", 0.1, 0.4) if params["POSITIVE_MIXUP"] else HPO_DEFAULTS["MIXUP_ALPHA"]
    return params


def objetivo(trial) -> float:
    spec = spec_de(sugerir(trial))
    name = plan.name(spec)
    trial.set_user_attr("experiment", name)
    trial.set_user_attr("params", json.dumps(dict(spec.params)))
    STORE.upload()  # el trial RUNNING queda en Comet: si se cae la VM se reintenta con los mismos parámetros
    PODA.update(trial=trial, best=float("-inf"))
    t0 = time.time()
    try:
        summary = plan.run(spec) or {}
    finally:
        PODA.update(trial=None)
        trial.set_user_attr("hours", (time.time() - t0) / 3600)
    image = metricas(plan, spec, "val", "image")
    breast = metricas(plan, spec, "val", "breast")
    trial.set_user_attr("val_roc_auc", image.get("roc_auc"))
    trial.set_user_attr("val_sens_at_spec_90", image.get("sens_at_spec_90"))
    trial.set_user_attr("val_breast_pr_auc", breast.get("pr_auc"))
    trial.set_user_attr("val_breast_roc_auc", breast.get("roc_auc"))
    trial.set_user_attr("comet_url", summary.get("comet_url"))
    return float(image["pr_auc"])


def specs_del_estudio(study_, states=None) -> list:
    specs = [spec_de(json.loads(t.user_attrs["params"])) for t in study_.trials if "params" in t.user_attrs and (states is None or t.state in states)]
    return plan.unique(specs)


def publicar_estado(study_, _trial=None) -> None:
    plan.publish_status(specs_del_estudio(study_))
    STORE.upload()


def tabla_trials(study_) -> pd.DataFrame:
    table = study_.trials_dataframe(attrs=("number", "value", "state", "params", "user_attrs", "duration"))
    return table.sort_values("value", ascending=False, na_position="last") if not table.empty else table


display(tabla_trials(study))

### Correr la búsqueda

Sigue hasta `HPO.N_TRIALS` trials terminados (completos + podados) o hasta gastar `HPO.HOURS` horas (suma de las horas de los trials, entre sesiones). Un trial que falla (por ejemplo, OOM) queda `FAIL` y la búsqueda sigue. Volver a ejecutar la celda continúa el estudio.

**Monitoreo**: la tabla de estado de los trials (pendientes, entrenando, cortados, terminados) es el asset `estado_matriz.csv` del experimento `estado_hpo`; cada trial loguea además `run_status` / `run_progress` como en la matriz y, si se poda, `hpo_status=pruned`.

In [ ]:
TERMINADOS = (TrialState.COMPLETE, TrialState.PRUNED)


def horas_usadas(study_) -> float:
    return float(sum(float(t.user_attrs.get("hours", 0.0)) for t in study_.trials if t.state in (*TERMINADOS, TrialState.FAIL)))


restantes = HPO["HOURS"] - horas_usadas(study)
hechos = sum(t.state in TERMINADOS for t in study.trials)
print(f"[hpo] {hechos}/{HPO['N_TRIALS']} trials terminados, {horas_usadas(study):.1f} h usadas, {max(restantes, 0):.1f} h restantes")
if restantes > 0.25 and hechos < HPO["N_TRIALS"]:
    study.optimize(
        objetivo,
        timeout=restantes * 3600,
        gc_after_trial=True,
        catch=(Exception,),
        callbacks=[MaxTrialsCallback(HPO["N_TRIALS"], states=TERMINADOS), publicar_estado],
    )
publicar_estado(study)
TRIALS = tabla_trials(study)
display(TRIALS)
print(f"Mejor trial: #{study.best_trial.number} val PR-AUC={study.best_value:.4f} -> {study.best_trial.user_attrs.get('experiment')}")

### Análisis del estudio

Historia, curvas intermedias (qué se podó y cuándo), importancia de cada hiperparámetro (fANOVA), coordenadas paralelas, slices y el contorno de los dos parámetros más importantes. Cada figura se sube como HTML al experimento `optuna_estudio`, junto con `trials.csv`. Con pocos trials completos las importancias son ruidosas: leerlas como tendencia, no como ranking.

In [ ]:
import optuna.visualization as ov


def log_text_asset(api_exp, text: str, name: str) -> None:
    """Como log_table_asset, para HTML o texto."""
    with tempfile.TemporaryDirectory() as tmp:
        path = Path(tmp) / name
        path.write_text(text, encoding="utf-8")
        api_exp.log_asset(str(path), name=name, overwrite=True)


FIGURAS = {}
for nombre, factory in {
    "historia": lambda: ov.plot_optimization_history(study),
    "intermedios": lambda: ov.plot_intermediate_values(study),
    "importancias": lambda: ov.plot_param_importances(study),
    "paralelas": lambda: ov.plot_parallel_coordinate(study),
    "slice": lambda: ov.plot_slice(study),
}.items():
    try:
        FIGURAS[nombre] = factory()
    except Exception as exc:
        print(f"[hpo] sin figura {nombre}: {exc}")
try:
    IMPORTANCIAS = optuna.importance.get_param_importances(study)
    display(pd.Series(IMPORTANCIAS, name="importancia").round(3))
    FIGURAS["contorno"] = ov.plot_contour(study, params=list(IMPORTANCIAS)[:2])
except Exception as exc:
    IMPORTANCIAS = {}
    print(f"[hpo] sin importancias todavía: {exc}")

for nombre, figura in FIGURAS.items():
    figura.show()
    try:
        log_text_asset(STORE.experiment(), figura.to_html(include_plotlyjs="cdn"), f"optuna_{nombre}.html")
    except Exception as exc:
        print(f"[hpo] no pude subir optuna_{nombre}.html ({exc})")
try:
    log_table_asset(STORE.experiment(), TRIALS, "trials.csv")
except Exception as exc:
    print(f"[hpo] no pude subir trials.csv ({exc})")

## Confirmación: etiquetas y semillas

El mejor config de la búsqueda (etiqueta A, semilla 42) se reentrena con las etiquetas A, B y C × semillas 42, 43 y 44. A/42 es el propio mejor trial y se saltea por nombre: son 8 corridas nuevas (~8 h). La etiqueta de entrenamiento es el único factor; la evaluación es siempre la tarea A sobre el val canónico completo.

| Etiqueta | Entrena con |
| --- | --- |
| A | 1–2 vs 3–5 |
| B | 1–2 vs 4–5 (sin BI-RADS 3 en train/val de la corrida) |
| C | 1–3 vs 4–5 |

Acá solo se mira **val**. El baseline es `resized_672_swint_seed{42,43,44}` de la matriz (`tesis_v4`); las diferencias van pareadas por semilla, con bootstrap agrupado por paciente.

In [ ]:
MEJOR = json.loads(study.best_trial.user_attrs["params"])
display(pd.Series(MEJOR, name="mejor config"))
CONFIRMACION = {f"label_{label}": [spec_de(MEJOR, seed=seed, label=label) for seed in SEMILLAS] for label in ETIQUETAS}
TODAS_CONF = plan.unique([spec for specs in CONFIRMACION.values() for spec in specs])
MEJOR_A = CONFIRMACION["label_A"][0]
assert plan.name(MEJOR_A) == study.best_trial.user_attrs["experiment"], "A/42 debería ser el mejor trial"

plan.run_all(TODAS_CONF, continue_on_error=True)
display(plan.status_table(TODAS_CONF))
for nombre, error in plan.failures.items():
    print(f"FALLO {nombre}: {error[:300]}")

In [ ]:
NIVELES = ["image", "breast"]


def reporte(plan_, specs, split, *, ci=False) -> pd.DataFrame:
    return pd.concat([plan_.evaluate(specs, split=split, level=nivel, ci=ci) for nivel in NIVELES], ignore_index=True)


def resumen_semillas(split) -> pd.DataFrame:
    tablas = [plan.seed_summary(CONFIRMACION, split=split, level=nivel) for nivel in NIVELES]
    tablas += [BASE_PLAN.seed_summary({"baseline_tesis_v4": BASE_SPECS}, split=split, level=nivel) for nivel in NIVELES]
    return pd.concat(tablas, ignore_index=True)


def pareado_vs_baseline(split, *, label="A") -> pd.DataFrame:
    """Por semilla: mejor config (etiqueta ``label``) - baseline, con IC 95 % pareado agrupado por paciente."""
    rows = []
    for nivel in NIVELES:
        for base_spec, spec in zip(BASE_SPECS, CONFIRMACION[f"label_{label}"]):
            if BASE_PLAN.lookup(base_spec) is None or plan.lookup(spec) is None:
                continue
            y, prob_base, groups = BASE_PLAN._level_arrays(BASE_PLAN.predictions(base_spec, split), PLAN["TASK"], nivel)
            y_alt, prob_alt, _ = plan._level_arrays(plan.predictions(spec, split), PLAN["TASK"], nivel)
            if not np.array_equal(y, y_alt):
                raise ValueError("baseline y mejor config no están alineados sobre las mismas imágenes")
            delta = PairedBootstrap.auc_delta(y, prob_base, prob_alt, groups=groups, samples=PLAN["BOOTSTRAP_SAMPLES"], seed=GENERAL["RANDOM_SEED"])
            rows.append({"split": split, "level": nivel, "label": label, "seed": base_spec.seed, "baseline": BASE_PLAN.name(base_spec), "mejor": plan.name(spec), **delta})
    return pd.DataFrame(rows)


CONF_VAL = pd.concat([reporte(plan, TODAS_CONF, "val"), reporte(BASE_PLAN, BASE_SPECS, "val")], ignore_index=True)
SEMILLAS_VAL = resumen_semillas("val")
PAREADO_VAL = pd.concat([pareado_vs_baseline("val", label=label) for label in ETIQUETAS], ignore_index=True)
display(CONF_VAL)
display(SEMILLAS_VAL)
display(PAREADO_VAL)
plan.publish_report(CONF_VAL, "confirmacion_val.csv")
plan.publish_report(SEMILLAS_VAL, "semillas_val.csv")
plan.publish_report(PAREADO_VAL, "pareado_vs_baseline_val.csv")

## Exprimir el modelo sin reentrenar

Todo se decide en **val** y se aplica a test una sola vez, en el reporte final.

- **TTA**: promedio de probabilidades sobre identidad, zoom central 0.92, desplazamientos de ±16 px y brillo/contraste ±5 %. Sin flip horizontal: las imágenes ya están lateralizadas.
- **Ensembles** (media de probabilidades): 3 semillas del mejor config con A; las etiquetas A+B+C en la semilla 42; las 9 corridas; los 3 mejores trials; y, para comparar a igual costo, las 3 semillas del baseline.
- **Agregación por mama**: máximo entre CC y MLO (la de `src`) vs media.
- **Temperature scaling** ajustado en val sobre la variante elegida: mejora Brier/ECE sin cambiar el ranking (AUC igual).

In [ ]:
from scipy.optimize import minimize_scalar

TTA_CACHE = {}


def desplazar(x, dx, dy):
    height, width = int(x.shape[1]), int(x.shape[2])
    x = tf.pad(x, [[0, 0], [max(dy, 0), max(-dy, 0)], [max(dx, 0), max(-dx, 0)], [0, 0]])
    return x[:, max(-dy, 0) : max(-dy, 0) + height, max(-dx, 0) : max(-dx, 0) + width, :]


def transformaciones_tta(size):
    return {
        "identidad": lambda x: x,
        "zoom_092": lambda x: tf.image.resize(tf.image.central_crop(x, 0.92), size),
        "dx_+16": lambda x: desplazar(x, 16, 0),
        "dx_-16": lambda x: desplazar(x, -16, 0),
        "dy_+16": lambda x: desplazar(x, 0, 16),
        "dy_-16": lambda x: desplazar(x, 0, -16),
        "brillo_+5": lambda x: tf.clip_by_value(x * 1.05, 0.0, 255.0),
        "contraste_-5": lambda x: tf.clip_by_value((x - tf.reduce_mean(x, axis=[1, 2, 3], keepdims=True)) * 0.95 + tf.reduce_mean(x, axis=[1, 2, 3], keepdims=True), 0.0, 255.0),
    }


def _sigmoid(z):
    z = np.asarray(z, dtype=np.float64)
    return 1.0 / (1.0 + np.exp(-z))


def predecir_tta(plan_, spec, split, *, batch_size=16) -> pd.DataFrame:
    """Predicciones del split canónico con TTA (columna ``y_prob`` = media; una columna por transformación)."""
    key = (plan_.name(spec), split)
    if key in TTA_CACHE:
        return TTA_CACHE[key]
    config = plan_._config_for(spec)
    builder = plan_._inference_model(spec)
    try:
        size = tuple(int(v) for v in builder.IMG_SIZE)
        provider = provider_with_hpo(config, size, batch_size, lateralize=True, mode="resized", cache_dataset=False)
        dataset = provider.build_eval(plan_.frames[split], name=f"tta_{split}")
        transforms = transformaciones_tta(size)
        probs = {name: [] for name in transforms}
        for xb, _ in as_tf_dataset(dataset.ordered()):
            xb = tf.cast(xb, tf.float32)
            for name, fn in transforms.items():
                probs[name].append(_sigmoid(np.asarray(builder.model.predict_on_batch(fn(xb))).reshape(-1)))
    finally:
        GpuResources.dispose_model_builder(builder)
        GpuResources.release(clear_keras_session=True)
    out = plan_.predictions(spec, split).copy()
    for name, chunks in probs.items():
        out[f"tta_{name}"] = np.concatenate(chunks)
    out["y_prob"] = out[[f"tta_{name}" for name in probs]].mean(axis=1)
    TTA_CACHE[key] = out
    return out


def ensemble(pares, split) -> pd.DataFrame:
    """Media de probabilidades de ``[(plan, spec), ...]`` sobre el split canónico (mismo orden de imágenes)."""
    tablas = [plan_.predictions(spec, split) for plan_, spec in pares]
    out = tablas[0].copy()
    for tabla in tablas[1:]:
        if not np.array_equal(tabla["image_id"].to_numpy(), out["image_id"].to_numpy()):
            raise ValueError("predicciones desalineadas")
    out["y_prob"] = np.mean([tabla["y_prob"].to_numpy() for tabla in tablas], axis=0)
    return out


def arrays(pred, level, how="max"):
    """``(y, prob, groups)`` de la tarea A por imagen o por mama (``how``: max | mean)."""
    if level == "image" or how == "max":
        return plan._level_arrays(pred, PLAN["TASK"], level)
    mask, y, _ = plan._task_arrays(pred, PLAN["TASK"])
    sub = pred.loc[mask].reset_index(drop=True)
    keys = BreastAggregation.keys_for(sub)
    table = sub[keys + (["patient_id"] if "patient_id" not in keys else [])].copy()
    table[keys] = table[keys].astype(str)
    table["y_true"] = y
    table["y_prob"] = sub["y_prob"].to_numpy()
    agg = {"y_true": "max", "y_prob": how}
    if "patient_id" not in keys:
        agg["patient_id"] = "first"
    grouped = table.groupby(keys, sort=True).agg(agg).reset_index()
    return grouped["y_true"].to_numpy(), grouped["y_prob"].to_numpy(), grouped["patient_id"].astype(str).to_numpy()


def ajustar_temperatura(y, prob) -> float:
    logit = np.log(np.clip(prob, 1e-7, 1 - 1e-7) / (1 - np.clip(prob, 1e-7, 1 - 1e-7)))

    def nll(t):
        p = np.clip(_sigmoid(logit / t), 1e-7, 1 - 1e-7)
        return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

    return float(minimize_scalar(nll, bounds=(0.05, 20.0), method="bounded").x)


def con_temperatura(prob, t):
    prob = np.clip(prob, 1e-7, 1 - 1e-7)
    return _sigmoid(np.log(prob / (1 - prob)) / t)


TOP3 = [spec_de(json.loads(t.user_attrs["params"])) for t in sorted((t for t in study.trials if t.state == TrialState.COMPLETE), key=lambda t: t.value, reverse=True)[:3]]
VARIANTES = {
    "baseline_seed42": lambda split: BASE_PLAN.predictions(BASE_SPECS[0], split),
    "ens_semillas_baseline": lambda split: ensemble([(BASE_PLAN, spec) for spec in BASE_SPECS if BASE_PLAN.lookup(spec) is not None], split),
    "mejor_A_seed42": lambda split: plan.predictions(MEJOR_A, split),
    "tta_mejor_A_seed42": lambda split: predecir_tta(plan, MEJOR_A, split),
    "ens_semillas_A": lambda split: ensemble([(plan, spec) for spec in CONFIRMACION["label_A"] if plan.lookup(spec) is not None], split),
    "ens_etiquetas_seed42": lambda split: ensemble([(plan, CONFIRMACION[f"label_{label}"][0]) for label in ETIQUETAS if plan.lookup(CONFIRMACION[f"label_{label}"][0]) is not None], split),
    "ens_todas_confirmacion": lambda split: ensemble([(plan, spec) for spec in TODAS_CONF if plan.lookup(spec) is not None], split),
    "ens_top3_trials": lambda split: ensemble([(plan, spec) for spec in TOP3], split),
}
NIVELES_POSTHOC = [("image", "max"), ("breast", "max"), ("breast", "mean")]


def evaluar_variantes(split, *, ci=False, temperaturas=None) -> pd.DataFrame:
    rows = []
    for nombre, fn in VARIANTES.items():
        try:
            pred, pred_val = fn(split), fn("val")
        except Exception as exc:
            print(f"[posthoc] {nombre}: {type(exc).__name__}: {exc}")
            continue
        for level, how in NIVELES_POSTHOC:
            y, prob, groups = arrays(pred, level, how)
            y_val, prob_val, _ = arrays(pred_val, level, how)
            variantes = [("", prob, prob_val)]
            t = (temperaturas or {}).get(nombre)
            if t is not None:
                variantes.append((f"temp_{t:.2f}", con_temperatura(prob, t), con_temperatura(prob_val, t)))
            for calibracion, p, p_val in variantes:
                m = ClinicalMetrics.compute(y, p, threshold=ThresholdSelector.youden_j(y_val, p_val))
                if ci:
                    m.update(ClinicalMetrics.bootstrap_ci(y, p, groups=groups, samples=PLAN["BOOTSTRAP_SAMPLES"], seed=GENERAL["RANDOM_SEED"]))
                rows.append({"variante": nombre, "split": split, "level": level, "agregacion_mama": how if level == "breast" else "", "calibracion": calibracion, **m})
    return pd.DataFrame(rows)


POSTHOC_VAL = evaluar_variantes("val")
por_imagen = POSTHOC_VAL[POSTHOC_VAL["level"] == "image"].set_index("variante")["pr_auc"]
por_mama = POSTHOC_VAL[POSTHOC_VAL["level"] == "breast"]
ELEGIDA = str(por_imagen.idxmax())
AGREGACION_ELEGIDA = str(por_mama[por_mama["variante"] == ELEGIDA].sort_values("pr_auc", ascending=False)["agregacion_mama"].iloc[0])
y_val_elegida, prob_val_elegida, _ = arrays(VARIANTES[ELEGIDA]("val"), "image")
TEMPERATURA = ajustar_temperatura(y_val_elegida, prob_val_elegida)
print(f"Elegida en val: {ELEGIDA} (PR-AUC imagen {por_imagen.max():.4f}); agregación por mama: {AGREGACION_ELEGIDA}; temperatura {TEMPERATURA:.2f}")
display(POSTHOC_VAL.sort_values(["level", "agregacion_mama", "pr_auc"], ascending=[True, True, False]))
plan.publish_report(POSTHOC_VAL, "posthoc_val.csv")

## Observabilidad: ¿mira las regiones correctas?

Para cada modelo (baseline de la matriz, mejor config con A, B y C; semilla 42), sobre **val**:

- **Mapas**:
  - Grad-CAM y Grad-CAM++ sobre la salida de Swin (21×21 a 672).
  - HiResCAM sobre el último bloque del stage 2 (42×42, el doble de resolución).
  - En un subconjunto: Integrated Gradients (baseline = imagen borrosa), SmoothGrad y oclusión deslizante (parche de 64 px, stride 48), que no depende de gradientes.
- **Localización contra la ROI anotada**. Se usan todas las cajas `pad_resized_*_norm` de cada imagen, del CSV crudo antes de deduplicar. La x se espeja en las mamas derechas, como hace el provider al lateralizar. Las cajas fuera de rango se recortan y las invertidas se descartan.
  - `pointing_game`: el máximo del mapa cae dentro de una caja (±15 px).
  - `energia_en_roi`: fracción de la energía del mapa dentro de las cajas. `energia_sobre_area` = esa fracción / área relativa de las cajas (1 = azar, >1 = enfocado).
  - `auroc_pixel`: el mapa como clasificador de píxeles dentro/fuera de la caja.
  - `iou_top10`: IoU del 10 % más alto del mapa con las cajas.
- **Atajos**: fracción de la energía fuera de la mama (componente principal de una máscara Otsu; incluye marcadores y texto) y en una banda del 5 % en el borde. Se mira sobre todo en negativos y falsos positivos.
- **Sanity check** (Adebayo et al., 2018): correlación de Spearman entre el Grad-CAM original y el del mismo modelo con la cabeza aleatorizada, y luego también el stage 3. Si el mapa casi no cambia (ρ alto), no está explicando al modelo.
- **Embeddings**: t-SNE de los vectores GAP de val, coloreados por BI-RADS, etiqueta A y tipo de error.
- **Galería**: falsos positivos y falsos negativos más extremos y verdaderos positivos más seguros (umbral de Youden en val), con overlay y caja ROI.
- **Subgrupos**: ROC/PR-AUC y sensibilidad/especificidad por densidad, vista, BI-RADS y tipo de hallazgo.

Un mapa de saliencia no prueba causalidad, y la ROI marca el hallazgo anotado, no necesariamente toda la evidencia. Por eso se cruzan métodos con gradiente, oclusión y el sanity check. Cada modelo loguea tablas, métricas y figuras en un experimento de Comet `observabilidad_{nombre}` (tags `hpo`, `observabilidad`). La tabla comparativa va como asset `observabilidad_resumen.csv` de `estado_hpo`.

In [ ]:
import cv2
from scipy.stats import spearmanr
from sklearn.manifold import TSNE
from sklearn.metrics import average_precision_score, roc_auc_score

from src.dataset.repository import DatasetRepository

ROI_COLUMNS = ("pad_resized_xmin_norm", "pad_resized_ymin_norm", "pad_resized_xmax_norm", "pad_resized_ymax_norm")
FLIP_SIDE = "R"  # lateralize_flip_side del provider
EPS = 1e-8


def cajas_por_imagen(raw: pd.DataFrame) -> dict:
    """Todas las cajas normalizadas por image_id, en el sistema de la imagen lateralizada."""
    out = {}
    sub = raw.dropna(subset=list(ROI_COLUMNS))
    for image_id, laterality, *box in sub[["image_id", "laterality", *ROI_COLUMNS]].itertuples(index=False):
        x0, y0, x1, y1 = (float(v) for v in box)
        if str(laterality) == FLIP_SIDE:
            x0, x1 = 1.0 - x1, 1.0 - x0
        x0, y0, x1, y1 = (float(np.clip(v, 0.0, 1.0)) for v in (x0, y0, x1, y1))
        if x1 - x0 <= 1e-3 or y1 - y0 <= 1e-3:
            continue
        out.setdefault(str(image_id), set()).add((round(x0, 5), round(y0, 5), round(x1, 5), round(y1, 5)))
    return {key: sorted(value) for key, value in out.items()}


CAJAS = cajas_por_imagen(DatasetRepository.download_and_build_dataset(CONFIG)["ds"])
print(f"{len(CAJAS)} imágenes con ROI válida")


def normalizar_mapa(m):
    m = np.asarray(m, dtype=np.float32)
    m = m - m.min()
    return m / (m.max() + EPS)


def mascara_cajas(cajas, height, width, margen=0):
    mask = np.zeros((height, width), dtype=bool)
    for x0, y0, x1, y1 in cajas:
        mask[max(0, int(y0 * height) - margen) : min(height, int(math.ceil(y1 * height)) + margen), max(0, int(x0 * width) - margen) : min(width, int(math.ceil(x1 * width)) + margen)] = True
    return mask


def localizacion(mapa, cajas) -> dict:
    height, width = mapa.shape
    box = mascara_cajas(cajas, height, width)
    box_tol = mascara_cajas(cajas, height, width, margen=15)
    area = float(box.mean())
    dentro = float((mapa * box).sum() / (mapa.sum() + EPS))
    iy, ix = np.unravel_index(int(np.argmax(mapa)), mapa.shape)
    small, small_box = mapa[::4, ::4], box[::4, ::4]
    auroc = float(roc_auc_score(small_box.ravel(), small.ravel())) if 0 < small_box.mean() < 1 else float("nan")
    top = mapa >= np.quantile(mapa, 0.90)
    return {
        "pointing_game": float(box_tol[iy, ix]),
        "energia_en_roi": dentro,
        "area_roi": area,
        "energia_sobre_area": dentro / max(area, EPS),
        "auroc_pixel": auroc,
        "iou_top10": float((top & box).sum() / max((top | box).sum(), 1)),
    }


def mascara_mama(img):
    gray = np.clip(np.asarray(img, dtype=np.float32).mean(-1), 0, 255).astype(np.uint8)
    gray = cv2.GaussianBlur(gray, (0, 0), 5)
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    n, labels, stats, _ = cv2.connectedComponentsWithStats((binary > 0).astype(np.uint8))
    if n <= 1:
        return binary > 0
    return labels == 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))


def atajos(mapa, img, borde=0.05) -> dict:
    mama = mascara_mama(img)
    energia = mapa.sum() + EPS
    height, width = mapa.shape
    band = np.zeros_like(mama)
    b = max(1, int(borde * min(height, width)))
    band[:b, :] = band[-b:, :] = band[:, :b] = band[:, -b:] = True
    return {"energia_fuera_mama": float((mapa * ~mama).sum() / energia), "energia_borde": float((mapa * band).sum() / energia), "fraccion_mama": float(mama.mean())}


class Explicador:
    """Mapas de saliencia de un resized_672_swint entrenado: forward manual preprocess -> Swin -> gap -> dense -> output."""

    CAPA_INTERMEDIA = "stage2_block5"  # último bloque del stage 2 (H/16)

    def __init__(self, builder):
        self.builder = builder
        self.model = builder.model
        self.backbone = builder.backbone
        self.preprocess = self.model.get_layer("preprocess_input")
        self.head = [self.model.get_layer(name) for name in ("gap", "dense", "dropout", "output")]
        self.features = keras.Model(self.backbone.inputs, [self.backbone.get_layer(self.CAPA_INTERMEDIA).output, self.backbone.output])

    @staticmethod
    def _espacial(t):
        t = tf.cast(t, tf.float32)
        if t.shape.rank == 3:
            side = int(round(int(t.shape[1]) ** 0.5))
            return tf.reshape(t, [-1, side, side, int(t.shape[-1])])
        return t

    def logits_from(self, fmap):
        x = fmap
        for layer in self.head:
            x = layer(x, training=False) if layer.name == "dropout" else layer(x)
        return tf.reshape(tf.cast(x, tf.float32), [-1])

    def forward(self, x):
        inter, fmap = self.features(self.preprocess(x), training=False)
        return inter, fmap, self.logits_from(fmap)

    def embeddings(self, x):
        _, fmap = self.features(self.preprocess(tf.cast(x, tf.float32)), training=False)
        fmap = self._espacial(fmap)
        return tf.reduce_mean(fmap, axis=[1, 2]).numpy(), _sigmoid(self.logits_from(fmap).numpy())

    def _up(self, cam, size):
        cam = tf.image.resize(cam[..., None], size, method="bilinear")[..., 0].numpy()
        return np.stack([normalizar_mapa(c) for c in cam])

    def cams(self, x):
        """Grad-CAM, Grad-CAM++ (salida de Swin) y HiResCAM (stage 2) para un batch; devuelve (mapas, probabilidades)."""
        x = tf.cast(x, tf.float32)
        with tf.GradientTape() as tape:
            inter, fmap, logit = self.forward(x)
            target = tf.reduce_sum(logit)
        g_inter, g_fmap = tape.gradient(target, [inter, fmap])
        A, G = self._espacial(fmap), self._espacial(g_fmap)
        Ai, Gi = self._espacial(inter), self._espacial(g_inter)
        gradcam = tf.nn.relu(tf.reduce_sum(tf.reduce_mean(G, axis=[1, 2], keepdims=True) * A, -1))
        g2, g3 = G**2, G**3
        denom = 2.0 * g2 + tf.reduce_sum(A, axis=[1, 2], keepdims=True) * g3
        alpha = g2 / tf.where(tf.abs(denom) > 1e-12, denom, tf.ones_like(denom))
        weights = tf.reduce_sum(alpha * tf.nn.relu(G), axis=[1, 2], keepdims=True)
        gradcampp = tf.nn.relu(tf.reduce_sum(weights * A, -1))
        hirescam = tf.nn.relu(tf.reduce_sum(Gi * Ai, -1))
        size = tuple(int(v) for v in x.shape[1:3])
        return {"gradcam": self._up(gradcam, size), "gradcam++": self._up(gradcampp, size), "hirescam_s2": self._up(hirescam, size)}, _sigmoid(logit.numpy())

    def _input_grads(self, xs):
        with tf.GradientTape() as tape:
            tape.watch(xs)
            target = tf.reduce_sum(self.forward(xs)[2])
        return tape.gradient(target, xs)

    def integrated_gradients(self, x, steps=32, chunk=8):
        x = tf.cast(x, tf.float32)[None]
        height, width = int(x.shape[1]), int(x.shape[2])
        base = tf.image.resize(tf.image.resize(x, [height // 16, width // 16], method="area"), [height, width])
        alphas = tf.linspace(0.0, 1.0, steps + 1)[1:]
        total = tf.zeros_like(x)
        for i in range(0, steps, chunk):
            a = tf.reshape(alphas[i : i + chunk], [-1, 1, 1, 1])
            total += tf.reduce_sum(self._input_grads(base + a * (x - base)), axis=0, keepdims=True)
        attribution = (x - base) * total / steps
        return normalizar_mapa(tf.reduce_sum(tf.abs(attribution), -1)[0].numpy())

    def smoothgrad(self, x, samples=16, chunk=8, sigma=0.10):
        x = tf.cast(x, tf.float32)[None]
        total = tf.zeros_like(x)
        for i in range(0, samples, chunk):
            n = min(chunk, samples - i)
            noisy = tf.repeat(x, n, axis=0) + tf.random.normal([n, *x.shape[1:]], stddev=sigma * 255.0, seed=GENERAL["RANDOM_SEED"] + i)
            total += tf.reduce_sum(tf.abs(self._input_grads(noisy)), axis=0, keepdims=True)
        mapa = tf.reduce_sum(total, -1)[0].numpy() / samples
        return normalizar_mapa(cv2.GaussianBlur(mapa, (0, 0), 4))

    def oclusion(self, x, patch=64, stride=48, batch=32):
        img = np.asarray(x, dtype=np.float32)
        height, width = img.shape[:2]
        base_logit = float(self.forward(tf.constant(img[None]))[2][0])
        fill = float(img.mean())
        coords = [(i, j) for i in range(0, height - patch + 1, stride) for j in range(0, width - patch + 1, stride)]
        heat, count = np.zeros((height, width), np.float32), np.zeros((height, width), np.float32)
        for k in range(0, len(coords), batch):
            chunk = coords[k : k + batch]
            imgs = np.repeat(img[None], len(chunk), axis=0)
            for n, (i, j) in enumerate(chunk):
                imgs[n, i : i + patch, j : j + patch, :] = fill
            logits = self.forward(tf.constant(imgs))[2].numpy()
            for n, (i, j) in enumerate(chunk):
                heat[i : i + patch, j : j + patch] += base_logit - logits[n]
                count[i : i + patch, j : j + patch] += 1
        return normalizar_mapa(np.maximum(heat / np.maximum(count, 1), 0))

    def aleatorizado(self, capas):
        """Context manager: reemplaza los pesos de ``capas`` por ruido con la misma escala y los restaura al salir."""
        explicador = self

        class _Ctx:
            def __enter__(self_inner):
                self_inner.saved = [(layer, layer.get_weights()) for layer in capas]
                rng = np.random.default_rng(GENERAL["RANDOM_SEED"])
                for layer, weights in self_inner.saved:
                    layer.set_weights([rng.normal(0.0, float(np.std(w)) + 1e-3, size=w.shape).astype(w.dtype) for w in weights])
                return explicador

            def __exit__(self_inner, *exc):
                for layer, weights in self_inner.saved:
                    layer.set_weights(weights)
                return False

        return _Ctx()


def _overlay(ax, img, mapa=None, cajas=(), titulo=""):
    gray = np.clip(np.asarray(img, dtype=np.float32).mean(-1), 0, 255) / 255.0
    ax.imshow(gray, cmap="gray")
    if mapa is not None:
        ax.imshow(mapa, cmap="jet", alpha=0.40, vmin=0, vmax=1)
    height, width = gray.shape
    for x0, y0, x1, y1 in cajas:
        ax.add_patch(plt.Rectangle((x0 * width, y0 * height), (x1 - x0) * width, (y1 - y0) * height, fill=False, edgecolor="lime", linewidth=1.5))
    ax.set_title(titulo, fontsize=8)
    ax.axis("off")

In [ ]:
def subgrupos(pred, frame, threshold) -> pd.DataFrame:
    """Discriminación y sensibilidad/especificidad (umbral de val) por subgrupo, tarea A por imagen."""
    mask, y, _ = plan._task_arrays(pred, PLAN["TASK"])
    tabla = pred.loc[mask].reset_index(drop=True).copy()
    tabla["y"] = y
    extra = frame.drop_duplicates("image_id").copy()
    extra.index = extra["image_id"].astype(str)
    columnas = [c for c in ("breast_density", "density", "view", "birads") if c in tabla.columns or c in extra.columns]
    hallazgos = [c for c in ("Mass", "Suspicious_Calcification", "Architectural_Distortion", "Asymmetry", "Focal_Asymmetry", "Suspicious_Lymph_Node") if c in extra.columns]
    for column in {*columnas, *hallazgos} - set(tabla.columns):
        tabla[column] = tabla["image_id"].astype(str).map(extra[column])
    rows = []

    def fila(nombre, valor, sub):
        pos, neg = int((sub["y"] == 1).sum()), int((sub["y"] == 0).sum())
        row = {"subgrupo": nombre, "valor": valor, "n": len(sub), "n_pos": pos}
        if pos and neg:
            row["roc_auc"] = float(roc_auc_score(sub["y"], sub["y_prob"]))
            row["pr_auc"] = float(average_precision_score(sub["y"], sub["y_prob"]))
        pred_pos = sub["y_prob"] >= threshold
        row["sensitivity"] = float(pred_pos[sub["y"] == 1].mean()) if pos else float("nan")
        row["specificity"] = float((~pred_pos)[sub["y"] == 0].mean()) if neg else float("nan")
        rows.append(row)

    for column in columnas:
        for valor, sub in tabla.groupby(column, dropna=False):
            fila(column, valor, sub)
    negativos = tabla[tabla["y"] == 0]
    for column in hallazgos:
        positivos = tabla[(tabla["y"] == 1) & (pd.to_numeric(tabla[column], errors="coerce") == 1)]
        fila("hallazgo", column, pd.concat([positivos, negativos]))
    return pd.DataFrame(rows)


def observar(plan_, spec, etiqueta, *, split="val", n_negativos=200, n_pixel=8, n_sanity=16, n_galeria=4) -> dict:
    """Mapas, localización vs ROI, atajos, sanity check, t-SNE, galería y subgrupos de un modelo; loguea todo en Comet."""
    name = plan_.name(spec)
    config = plan_._config_for(spec)
    pred = plan_.predictions(spec, split).reset_index(drop=True)
    y_val, prob_val, _ = plan_._level_arrays(plan_.predictions(spec, "val"), plan_.task, "image")
    threshold = ThresholdSelector.youden_j(y_val, prob_val)
    frame = plan_.frames[split]
    birads = pd.to_numeric(pred["birads"], errors="coerce")
    ids = pred["image_id"].astype(str)
    positivos = pred.index[(birads >= 3) & ids.isin(CAJAS)].to_numpy()
    negativos = pred.index[birads <= 2].to_numpy()
    rng = np.random.default_rng(GENERAL["RANDOM_SEED"])
    negativos_obs = rng.choice(negativos, size=min(n_negativos, len(negativos)), replace=False) if len(negativos) else negativos
    prob = pred["y_prob"].to_numpy()
    fn = positivos[np.argsort(prob[positivos])][:n_galeria]
    tp = positivos[np.argsort(-prob[positivos])][:n_galeria]
    fp = negativos[np.argsort(-prob[negativos])][:n_galeria]
    pixel = rng.choice(positivos, size=min(n_pixel, len(positivos)), replace=False) if len(positivos) else positivos
    sanity = set(positivos[:n_sanity].tolist())
    galeria = {"FN": fn.tolist(), "TP": tp.tolist(), "FP": fp.tolist()}
    guardar = {*fn.tolist(), *tp.tolist(), *fp.tolist(), *pixel.tolist()}
    indices = np.unique(np.concatenate([positivos, negativos_obs, fp])).astype(int)

    builder = plan_._inference_model(spec)
    experiment = comet_ml.start(project_name=COMET["PROJECT_NAME"], experiment_config=comet_ml.ExperimentConfig(name=f"observabilidad_{name}", tags=["hpo", "observabilidad"], auto_metric_logging=False))
    try:
        experiment.log_parameters({"experiment": name, "etiqueta": etiqueta, "split": split, "threshold_youden_val": threshold, "n_positivos_roi": len(positivos), "n_negativos": len(negativos_obs)})
        explicador = Explicador(builder)
        size = tuple(int(v) for v in builder.IMG_SIZE)
        provider = provider_with_hpo(config, size, 8, lateralize=True, mode="resized", cache_dataset=False)
        por_id = frame.copy()
        por_id.index = por_id["image_id"].astype(str)
        dataset = provider.build_eval(por_id.loc[pred.loc[indices, "image_id"].astype(str)].reset_index(drop=True), name=f"obs_{split}")
        pos_set = set(positivos.tolist())
        loc_rows, atajo_rows, guardadas, sanity_maps = [], [], {}, {}
        cursor = 0
        for xb, _ in as_tf_dataset(dataset.ordered()):
            mapas, probs = explicador.cams(xb)
            imgs = np.asarray(xb, dtype=np.float32)
            for k in range(len(imgs)):
                idx = int(indices[cursor + k])
                row = pred.loc[idx]
                es_pos = idx in pos_set
                base = {"image_id": str(row["image_id"]), "birads": row["birads"], "prob": float(probs[k]), "pred_pos": bool(probs[k] >= threshold), "positivo": es_pos}
                base["resultado"] = ("TP" if base["pred_pos"] else "FN") if es_pos else ("FP" if base["pred_pos"] else "TN")
                for metodo, m in mapas.items():
                    if es_pos:
                        loc_rows.append({**base, "metodo": metodo, **localizacion(m[k], CAJAS[base["image_id"]])})
                    atajo_rows.append({**base, "metodo": metodo, **atajos(m[k], imgs[k])})
                if idx in guardar:
                    guardadas[idx] = (imgs[k].astype(np.uint8), {metodo: m[k] for metodo, m in mapas.items()})
                if idx in sanity:
                    sanity_maps[idx] = (imgs[k], mapas["gradcam"][k])
            cursor += len(imgs)
        LOC = pd.DataFrame(loc_rows)
        ATAJOS = pd.DataFrame(atajo_rows)

        # Métodos por píxel (más caros) sobre un subconjunto de positivos con ROI.
        pixel_rows, pixel_maps = [], {}
        for idx in pixel.tolist():
            img, maps = guardadas[int(idx)]
            extra = {"integrated_gradients": explicador.integrated_gradients(img), "smoothgrad": explicador.smoothgrad(img), "oclusion": explicador.oclusion(img)}
            pixel_maps[int(idx)] = {**maps, **extra}
            for metodo, m in extra.items():
                pixel_rows.append({"image_id": str(pred.loc[idx, "image_id"]), "metodo": metodo, **localizacion(m, CAJAS[str(pred.loc[idx, "image_id"])])})
        LOC = pd.concat([LOC, pd.DataFrame(pixel_rows)], ignore_index=True)

        # Sanity check por aleatorización en cascada: cabeza, después también el stage 3.
        sanity_rows = []
        cabeza = explicador.head[1:]
        stage3 = [layer for layer in explicador.backbone.layers if layer.name.startswith("stage3") or layer.name == "encoder_norm"]
        for nivel, capas in (("cabeza", cabeza), ("cabeza+stage3", cabeza + stage3)):
            with explicador.aleatorizado(capas):
                for idx, (img, original) in sanity_maps.items():
                    mapas, _ = explicador.cams(img[None])
                    rho = spearmanr(original[::4, ::4].ravel(), mapas["gradcam"][0][::4, ::4].ravel()).correlation
                    sanity_rows.append({"image_id": str(pred.loc[idx, "image_id"]), "aleatorizado": nivel, "spearman": float(rho)})
        SANITY = pd.DataFrame(sanity_rows)

        # Embeddings (GAP) del split completo -> t-SNE.
        full = provider.build_eval(frame, name=f"emb_{split}")
        feats = np.concatenate([explicador.embeddings(xb)[0] for xb, _ in as_tf_dataset(full.ordered())])
        coords = TSNE(n_components=2, init="pca", perplexity=30, random_state=GENERAL["RANDOM_SEED"]).fit_transform(feats)
        y_task = (birads >= 3).astype(int).to_numpy()
        pred_pos = prob >= threshold
        error = np.where(y_task == 1, np.where(pred_pos, "TP", "FN"), np.where(pred_pos, "FP", "TN"))
        fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
        sc = axes[0].scatter(coords[:, 0], coords[:, 1], c=birads.to_numpy(), cmap="viridis", s=6)
        fig.colorbar(sc, ax=axes[0], label="BI-RADS")
        axes[1].scatter(coords[:, 0], coords[:, 1], c=y_task, cmap="coolwarm", s=6)
        for tipo, color in (("TN", "lightgray"), ("TP", "tab:green"), ("FP", "tab:orange"), ("FN", "tab:red")):
            sel = error == tipo
            axes[2].scatter(coords[sel, 0], coords[sel, 1], s=6, c=color, label=f"{tipo} ({sel.sum()})")
        axes[2].legend(markerscale=3)
        for ax, title in zip(axes, ("BI-RADS", "etiqueta A (3-5)", "resultado (Youden val)")):
            ax.set_title(f"{etiqueta}: {title}")
            ax.set_xticks([])
            ax.set_yticks([])
        experiment.log_figure(figure_name="tsne_gap", figure=fig)
        plt.show()
        plt.close(fig)

        # Galería FN / TP / FP con Grad-CAM y HiResCAM.
        filas = [(tipo, idx) for tipo, lista in galeria.items() for idx in lista if idx in guardadas]
        if filas:
            fig, axes = plt.subplots(len(filas), 3, figsize=(9, 3 * len(filas)), squeeze=False)
            for r, (tipo, idx) in enumerate(filas):
                img, maps = guardadas[idx]
                image_id = str(pred.loc[idx, "image_id"])
                cajas = CAJAS.get(image_id, ())
                titulo = f"{tipo} BI-RADS {pred.loc[idx, 'birads']} p={prob[idx]:.2f}"
                _overlay(axes[r, 0], img, None, cajas, titulo)
                _overlay(axes[r, 1], img, maps["gradcam"], cajas, "Grad-CAM")
                _overlay(axes[r, 2], img, maps["hirescam_s2"], cajas, "HiResCAM stage 2")
            fig.tight_layout()
            experiment.log_figure(figure_name="galeria_errores", figure=fig)
            plt.show()
            plt.close(fig)

        # Comparación de métodos sobre el subconjunto por píxel.
        if pixel_maps:
            metodos = ["gradcam", "gradcam++", "hirescam_s2", "integrated_gradients", "smoothgrad", "oclusion"]
            fig, axes = plt.subplots(len(pixel_maps), len(metodos) + 1, figsize=(2.6 * (len(metodos) + 1), 2.8 * len(pixel_maps)), squeeze=False)
            for r, (idx, maps) in enumerate(pixel_maps.items()):
                img = guardadas[idx][0]
                cajas = CAJAS[str(pred.loc[idx, "image_id"])]
                _overlay(axes[r, 0], img, None, cajas, f"BI-RADS {pred.loc[idx, 'birads']} p={prob[idx]:.2f}")
                for c, metodo in enumerate(metodos, start=1):
                    _overlay(axes[r, c], img, maps[metodo], cajas, metodo)
            fig.tight_layout()
            experiment.log_figure(figure_name="metodos_saliencia", figure=fig)
            plt.show()
            plt.close(fig)

        SUBGRUPOS = subgrupos(pred, frame, threshold)
        resumen = LOC.drop(columns=[c for c in ("prob", "pred_pos", "positivo", "birads") if c in LOC.columns]).groupby("metodo").mean(numeric_only=True).add_prefix("loc_")
        resumen = resumen.join(ATAJOS[ATAJOS["resultado"].isin(["FP", "TN"])].groupby("metodo")[["energia_fuera_mama", "energia_borde"]].mean().add_prefix("neg_"), how="outer")
        resumen = resumen.reset_index().assign(modelo=etiqueta, experiment=name)
        sanity_resumen = SANITY.groupby("aleatorizado")["spearman"].agg(["mean", "std"]).reset_index().assign(modelo=etiqueta) if not SANITY.empty else SANITY
        for tabla, archivo in ((LOC, "localizacion.csv"), (ATAJOS, "atajos.csv"), (SANITY, "sanity.csv"), (SUBGRUPOS, "subgrupos.csv"), (resumen, "resumen.csv")):
            if not tabla.empty:
                experiment.log_table(archivo, tabular_data=tabla)
        for row in resumen.itertuples(index=False):
            metodo = row.metodo.replace("+", "pp")
            experiment.log_metrics({f"{metodo}_{key}": float(value) for key, value in row._asdict().items() if key not in ("metodo", "modelo", "experiment") and isinstance(value, (int, float)) and np.isfinite(value)})
        if not SANITY.empty:
            experiment.log_metrics({f"sanity_spearman_{r.aleatorizado.replace('+', '_')}": float(r.mean) for r in sanity_resumen.itertuples(index=False)})
        display(resumen)
        display(sanity_resumen)
        display(LOC.groupby(["metodo", "resultado"]).mean(numeric_only=True).round(3) if "resultado" in LOC.columns else LOC)
        display(SUBGRUPOS)
        return {"localizacion": LOC, "atajos": ATAJOS, "sanity": SANITY, "subgrupos": SUBGRUPOS, "resumen": resumen, "sanity_resumen": sanity_resumen}
    finally:
        try:
            experiment.end()
        except Exception:
            pass
        GpuResources.dispose_model_builder(builder)
        GpuResources.release(clear_keras_session=True)

In [ ]:
MODELOS_OBS = {
    "baseline_tesis_v4": (BASE_PLAN, BASE_SPECS[0]),
    **{f"mejor_{label}": (plan, CONFIRMACION[f"label_{label}"][0]) for label in ETIQUETAS},
}
OBS = {}
for etiqueta, (plan_, spec) in MODELOS_OBS.items():
    if plan_.lookup(spec) is None:
        print(f"[obs] {plan_.name(spec)} todavía no terminó; se saltea")
        continue
    print(f"\n=== {etiqueta}: {plan_.name(spec)} ===")
    OBS[etiqueta] = observar(plan_, spec, etiqueta)

OBS_RESUMEN = pd.concat([o["resumen"] for o in OBS.values()], ignore_index=True)
OBS_SANITY = pd.concat([o["sanity_resumen"] for o in OBS.values() if not o["sanity_resumen"].empty], ignore_index=True)
display(OBS_RESUMEN.pivot_table(index="metodo", columns="modelo", values=["loc_pointing_game", "loc_energia_sobre_area", "loc_auroc_pixel", "neg_energia_fuera_mama"]).round(3))
display(OBS_SANITY)
plan.publish_report(OBS_RESUMEN, "observabilidad_resumen.csv")
plan.publish_report(OBS_SANITY, "observabilidad_sanity.csv")

## Reporte final en test (una sola vez)

Con todo decidido en val (mejor config, variante post-hoc, agregación por mama y temperatura):

- Las 9 corridas de confirmación y las 3 del baseline, por imagen y por mama, con IC 95 % agrupado por paciente.
- Media ± desvío entre semillas por etiqueta y del baseline.
- Diferencia pareada contra el baseline, semilla por semilla.
- Todas las variantes post-hoc con IC. La columna `elegida_en_val` marca la que se eligió antes de mirar test, y la variante elegida se repite con temperature scaling.

Todo se sube como assets de `estado_hpo` (`confirmacion_test.csv`, `semillas_test.csv`, `pareado_vs_baseline_test.csv`, `reporte_final_test.csv`).

In [ ]:
CONF_TEST = pd.concat([reporte(plan, TODAS_CONF, "test", ci=True), reporte(BASE_PLAN, BASE_SPECS, "test", ci=True)], ignore_index=True)
SEMILLAS_TEST = resumen_semillas("test")
PAREADO_TEST = pd.concat([pareado_vs_baseline("test", label=label) for label in ETIQUETAS], ignore_index=True)
FINAL_TEST = evaluar_variantes("test", ci=True, temperaturas={ELEGIDA: TEMPERATURA})
FINAL_TEST.insert(1, "elegida_en_val", (FINAL_TEST["variante"] == ELEGIDA) & ((FINAL_TEST["level"] == "image") | (FINAL_TEST["agregacion_mama"] == AGREGACION_ELEGIDA)))
display(CONF_TEST)
display(SEMILLAS_TEST)
display(PAREADO_TEST)
display(FINAL_TEST.sort_values(["level", "agregacion_mama", "pr_auc"], ascending=[True, True, False]))
plan.publish_report(CONF_TEST, "confirmacion_test.csv")
plan.publish_report(SEMILLAS_TEST, "semillas_test.csv")
plan.publish_report(PAREADO_TEST, "pareado_vs_baseline_test.csv")
plan.publish_report(FINAL_TEST, "reporte_final_test.csv")